In [ ]:
# In[1]:

import pandas as pd

# Read telemetry files and parse timestamps as UTC datetimes
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregation: group by (cmdb_id, kpi_name) -> count & global P95 of 'value'
metric_summary = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count', global_p95_value=lambda x: x.quantile(0.95))
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
# keep desired column order and round p95 for compactness
metric_summary['global_p95_value'] = metric_summary['global_p95_value'].round(6)
metric_out = metric_summary[['cmdb_id', 'kpi_name', 'count', 'global_p95_value']].head(50)

# 2) trace.csv: unique (cmdb_id, trace_name) pairs with record counts
trace_summary = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
trace_out = trace_summary[['cmdb_id', 'trace_name', 'count']].head(50)

# 3) log.csv: unique (cmdb_id, log_name) pairs with record counts
log_summary = (
    log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
log_out = log_summary[['cmdb_id', 'log_name', 'count']].head(50)

# 4) error_logs.csv: counts per cmdb_id and up to 10 example messages (truncated)
# create examples joined by " || ", each message truncated to 200 chars
examples_series = (
    error_logs
    .groupby('cmdb_id')['message']
    .apply(lambda s: ' || '.join(s.astype(str).str.slice(0,200).head(10).tolist()))
)
counts_series = error_logs.groupby('cmdb_id').size().rename('count')

error_summary = (
    pd.concat([counts_series, examples_series.rename('examples')], axis=1)
    .reset_index()
    .sort_values('count', ascending=False)
)
# keep compact: show up to all cmdb_id rows but examples truncated already; limit display to top 50 for safety
error_out = error_summary[['cmdb_id', 'count', 'examples']].head(50)

# Display outputs (as DataFrames). Use variable names as the final evaluation.
metric_out, trace_out, log_out, error_out

```
Out[1]:
```
Summary of results (plain English):

- metric.csv
  - Produced the requested aggregation by (cmdb_id, kpi_name) with count and global P95. I returned the top 50 rows sorted by count.
  - Most metric series have count = 25 (the dataset is per-minute normalized series); examples of P95 values:
    - carts: cpu P95 ≈ 1.690491, mem P95 ≈ 2.166387e+08
    - queue-master: diskio P95 ≈ 8.015722e+04, mem P95 ≈ 2.716967e+08, socket P95 = 3.0
    - rabbitmq: cpu P95 ≈ 0.898781
  - In short: metrics are present and P95s computed for up to 50 most-populated (cmdb_id, kpi) pairs.

- trace.csv
  - No trace records were present. The trace summary is empty.

- log.csv
  - Unique (cmdb_id, log_name) pairs were found (22 pairs in the result). Top examples (count = 25 each): catalogue.log.error_count, catalogue.log.row_count, orders.log.row_count, orders.log.error_count, front-end.log.row_count. Many services have full 25-sample log series.

- error_logs.csv
  - No error log records were present; the error logs summary is empty.

Overall takeaway: metric and log telemetry are available and summarized; trace and error_log telemetry are absent in this dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count  global_p95_value
0          carts         cpu     25      1.690491e+00
1          carts      diskio     25      0.000000e+00
2          carts  latency-50     25      1.727000e-02
3          carts  latency-90     25      3.898600e-02
4          carts         mem     25      2.166387e+08
..           ...         ...    ...               ...
45      shipping  latency-50     25      3.161000e-03
46  queue-master      diskio     25      8.015722e+04
47  queue-master         mem     25      2.716967e+08
48  queue-master      socket     25      3.000000e+00
49      rabbitmq         cpu     25      8.987810e-01

[50 rows x 4 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [],        cmdb_id         log_name  count
0    catalogue  log.error_count     25
1    catalogue    log.row_count     25
2       orders    log.row_count     25
3       orders  log.error_count     25
4    front-end    log.row_count     25
..         ...              ...    ...
17       carts    log.row_count     19
18  session-db    log.row_count      5
19  session-db  log.error_count      5
20   orders-db  log.error_count      1
21   orders-db    log.row_count      1

[22 rows x 3 columns], Empty DataFrame
Columns: [cmdb_id, count, examples]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing metric DataFrame if present; otherwise read from CSV
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed (in case metric was loaded earlier without parsing)
if metric['timestamp'].dtype == 'int64' or metric['timestamp'].dtype == 'float64':
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full series (must be done BEFORE window filtering)
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
)

# 2) Define incident window in UTC and filter metric rows within the window
start_ts = pd.Timestamp("2024-01-21 06:28:36", tz="UTC")
end_ts   = pd.Timestamp("2024-01-21 06:58:36", tz="UTC")

window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window rows so we can mark anomalies using global thresholds
if not window.empty:
    window = window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    window['is_high'] = window['value'] >= window['global_p95']
    window['is_low']  = window['value'] <= window['global_p05']
else:
    # create an empty window with thresholds columns to allow downstream merges
    window = pd.DataFrame(columns=list(metric.columns) + ['global_p95', 'global_p05', 'is_high', 'is_low'])

# 4) Aggregate anomaly info in the window per (cmdb_id, kpi_name)
if not window.empty:
    agg_counts = (
        window
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            window_count=('value', 'count'),
            anomaly_high_count=('is_high', 'sum'),
            anomaly_low_count=('is_low', 'sum')
        )
    )
    # earliest timestamps where conditions met
    def earliest_ts(subdf, flag_col):
        sub = subdf[subdf[flag_col]]
        if sub.empty:
            return pd.NaT
        return sub['timestamp'].min()
    earliest_high = (
        window.groupby(['cmdb_id', 'kpi_name']).apply(lambda g: earliest_ts(g, 'is_high'))
        .reset_index().rename(columns={0: 'earliest_high_timestamp_utc'})
    )
    earliest_low = (
        window.groupby(['cmdb_id', 'kpi_name']).apply(lambda g: earliest_ts(g, 'is_low'))
        .reset_index().rename(columns={0: 'earliest_low_timestamp_utc'})
    )

    agg = thresholds.merge(agg_counts, on=['cmdb_id', 'kpi_name'], how='left')
    agg = agg.merge(earliest_high, on=['cmdb_id', 'kpi_name'], how='left')
    agg = agg.merge(earliest_low, on=['cmdb_id', 'kpi_name'], how='left')
else:
    # No rows in window: produce table with zero counts and NaT earliest timestamps for all threshold groups
    agg = thresholds.copy()
    agg['window_count'] = 0
    agg['anomaly_high_count'] = 0
    agg['anomaly_low_count'] = 0
    agg['earliest_high_timestamp_utc'] = pd.NaT
    agg['earliest_low_timestamp_utc'] = pd.NaT

# 5) Fill NaNs for groups not present in window
agg['window_count'] = agg['window_count'].fillna(0).astype(int)
agg['anomaly_high_count'] = agg['anomaly_high_count'].fillna(0).astype(int)
agg['anomaly_low_count'] = agg['anomaly_low_count'].fillna(0).astype(int)

# Format earliest timestamps as ISO strings (UTC) or None
def fmt_ts(col):
    return col.dt.strftime('%Y-%m-%dT%H:%M:%SZ').where(col.notna())
agg['earliest_high_timestamp_utc'] = fmt_ts(pd.to_datetime(agg['earliest_high_timestamp_utc'], utc=True))
agg['earliest_low_timestamp_utc']  = fmt_ts(pd.to_datetime(agg['earliest_low_timestamp_utc'], utc=True))

# 6) Round global thresholds for compactness
agg['global_p95'] = agg['global_p95'].round(6)
agg['global_p05'] = agg['global_p05'].round(6)

# 7) Select and order columns as requested
result = agg[['cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
              'window_count', 'anomaly_high_count', 'earliest_high_timestamp_utc',
              'anomaly_low_count', 'earliest_low_timestamp_utc']].copy()

# 8) Sort by total anomalies desc and limit to top 20
result['total_anomalies'] = result['anomaly_high_count'] + result['anomaly_low_count']
result = result.sort_values(by=['total_anomalies', 'window_count'], ascending=[False, False]).drop(columns=['total_anomalies']).reset_index(drop=True)
result = result.head(20)

# Display compact CSV-like table (as a DataFrame variable)
result

```
Out[2]:
```
Summary (plain English):

Method: global P95 and P05 were computed from the full metric series for each (cmdb_id, kpi_name). Metrics were then filtered to the incident window 2024-01-21 06:28:36–06:58:36 UTC and counts of high/low anomalies (value ≥ global_p95, value ≤ global_p05) were produced per (cmdb_id, kpi_name).

Top findings (sorted by total anomalies):

- Many entries show the maximum counts within the window (window_count = 25) and equal high/low anomaly counts of 25 each. Examples: 
  - carts — diskio (25 high, 25 low; earliest high/low 2024-01-21T06:31:00Z)
  - carts-db — socket (25 high, 25 low; earliest 2024-01-21T06:31:00Z)
  - catalogue-db — diskio and catalogue-db — socket (each 25 high, 25 low; earliest 06:31:00Z)
  - front-end — error, payment — socket, queue-master — socket, rabbitmq — diskio, rabbitmq — socket, rabbitmq-exporter — mem and socket, session-db — socket, user — error, user-db — socket (all with 25/25 and earliest 06:31:00Z)

- Some KPIs show imbalanced anomaly counts (suggesting more low or more high occurrences):
  - orders-db — socket: window_count 25, anomaly_high_count 11, anomaly_low_count 13 (earliest high 06:31:00Z, earliest low 06:43:00Z)
  - user — socket: window_count 25, anomaly_high_count 2, anomaly_low_count 22 (earliest high 06:54:00Z, earliest low 06:31:00Z)
  - catalogue — socket: window_count 25, anomaly_high_count 11, anomaly_low_count 12 (earliest high 06:45:00Z, earliest low 06:31:00Z)
  - session-db — diskio: window_count 25, anomaly_high_count 2, anomaly_low_count 15 (earliest high 06:34:00Z, earliest low 06:31:00Z)
  - rabbitmq — mem: window_count 25, anomaly_high_count 2, anomaly_low_count 4 (earliest high 06:54:00Z, earliest low 06:34:00Z)

Takeaway / next steps:
- Multiple services and KPIs (notably socket, diskio, memory and some error metrics) show anomalies during the incident window. The earliest anomaly timestamps cluster around 2024-01-21T06:31:00Z for many series.
- Prioritize investigation of socket and diskio KPIs for services: carts, catalogue (and catalogue-db), rabbitmq (and rabbitmq-exporter), session-db, user, and orders-db — these show the highest anomaly counts in the window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name    global_p95  global_p05  window_count  anomaly_high_count earliest_high_timestamp_utc  anomaly_low_count earliest_low_timestamp_utc
0               carts   diskio  0.000000e+00         0.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
1            carts-db   socket  8.000000e+00         8.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
2        catalogue-db   diskio  0.000000e+00         0.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
3        catalogue-db   socket  4.000000e+00         4.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
4           front-end    error  0.000000e+00         0.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
5             payment   socket  5.000000e+00         5.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
6        queue-master   socket  3.000000e+00         3.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
7            rabbitmq   diskio  0.000000e+00         0.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
8            rabbitmq   socket  1.100000e+01        11.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
9   rabbitmq-exporter      mem  4.538368e+06   4538368.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
10  rabbitmq-exporter   socket  1.000000e+00         1.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
11         session-db   socket  3.000000e+00         3.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
12               user    error  0.000000e+00         0.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
13            user-db   socket  8.000000e+00         8.0            25                  25        2024-01-21T06:31:00Z                 25       2024-01-21T06:31:00Z
14          catalogue   diskio  0.000000e+00         0.0            24                  24        2024-01-21T06:32:00Z                 24       2024-01-21T06:32:00Z
15          orders-db   socket  7.000000e+00         5.0            25                  11        2024-01-21T06:31:00Z                 13       2024-01-21T06:43:00Z
16               user   socket  1.784333e+01        17.0            25                   2        2024-01-21T06:54:00Z                 22       2024-01-21T06:31:00Z
17          catalogue   socket  1.000000e+01         6.0            25                  11        2024-01-21T06:45:00Z                 12       2024-01-21T06:31:00Z
18         session-db   diskio  1.628708e+04         0.0            25                   2        2024-01-21T06:34:00Z                 15       2024-01-21T06:31:00Z
19           rabbitmq      mem  9.820601e+07  95191040.0            25                   2        2024-01-21T06:54:00Z                  4       2024-01-21T06:34:00Z```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing 'log' if present, otherwise read and parse timestamps
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure timestamp column is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(log['timestamp']):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Define incident window (UTC)
start_ts = pd.Timestamp("2024-01-21 06:28:36", tz="UTC")
end_ts   = pd.Timestamp("2024-01-21 06:58:36", tz="UTC")

# Filter to window
log_window = log[(log['timestamp'] >= start_ts) & (log['timestamp'] <= end_ts)].copy()

# Filter to log_name values that contain 'error' or 'row_count' (case-sensitive)
mask_names = log_window['log_name'].str.contains('error|row_count', regex=True, na=False)
log_window_relevant = log_window[mask_names].copy()

# If no relevant rows, produce empty DataFrames with expected columns
if log_window_relevant.empty:
    log_group_out = pd.DataFrame(columns=['cmdb_id','log_name','window_count','sum_value','mean_value','earliest_pos_timestamp_utc'])
    cmdb_agg_out = pd.DataFrame(columns=['cmdb_id','total_error_sum','mean_row_count'])
else:
    # Aggregate per (cmdb_id, log_name)
    grp = (
        log_window_relevant
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(window_count=('value', 'count'),
             sum_value=('value', 'sum'),
             mean_value=('value', 'mean'))
    )

    # Earliest timestamp in window where value > 0 per group
    earliest_pos = (
        log_window_relevant[log_window_relevant['value'] > 0]
        .groupby(['cmdb_id', 'log_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp': 'earliest_pos_timestamp_utc'})
    )

    # Merge earliest timestamps
    log_group_out = grp.merge(earliest_pos, on=['cmdb_id', 'log_name'], how='left')

    # Format earliest timestamps as ISO UTC strings or None
    log_group_out['earliest_pos_timestamp_utc'] = pd.to_datetime(log_group_out['earliest_pos_timestamp_utc'], utc=True)
    log_group_out['earliest_pos_timestamp_utc'] = log_group_out['earliest_pos_timestamp_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    log_group_out['earliest_pos_timestamp_utc'] = log_group_out['earliest_pos_timestamp_utc'].where(log_group_out['earliest_pos_timestamp_utc'].notna(), None)

    # Round numeric columns for compactness
    log_group_out['sum_value'] = log_group_out['sum_value'].round(6)
    log_group_out['mean_value'] = log_group_out['mean_value'].round(6)

    # Sort by sum_value desc and limit to top 20
    log_group_out = log_group_out.sort_values(by='sum_value', ascending=False).reset_index(drop=True).head(20)

    # Now compute per-cmdb_id aggregates over those log_names:
    # total_error_sum: sum of values where log_name contains 'error'
    error_rows = log_window_relevant[log_window_relevant['log_name'].str.contains('error', regex=False, na=False)]
    total_error_sum = error_rows.groupby('cmdb_id', as_index=False)['value'].sum().rename(columns={'value': 'total_error_sum'})

    # mean_row_count: mean of values where log_name == 'log.row_count'
    rowcount_rows = log_window_relevant[log_window_relevant['log_name'] == 'log.row_count']
    mean_row_count = rowcount_rows.groupby('cmdb_id', as_index=False)['value'].mean().rename(columns={'value': 'mean_row_count'})

    # Merge per-cmdb_id aggregates (use outer join to include cmdb_ids present in either)
    cmdb_agg_out = pd.merge(total_error_sum, mean_row_count, on='cmdb_id', how='outer')

    # If some cmdb_id present in relevant logs but not in either aggregate, ensure they appear with NaNs filled appropriately
    all_cmdbs = pd.DataFrame({'cmdb_id': log_window_relevant['cmdb_id'].unique()})
    cmdb_agg_out = all_cmdbs.merge(cmdb_agg_out, on='cmdb_id', how='left')

    # Fill missing total_error_sum with 0 (no error logs in window) and round mean_row_count
    cmdb_agg_out['total_error_sum'] = cmdb_agg_out['total_error_sum'].fillna(0).round(6)
    cmdb_agg_out['mean_row_count'] = cmdb_agg_out['mean_row_count'].round(6)

# Final output variables (compact tables). Use variable names as display.
log_group_out, cmdb_agg_out

```
Out[3]:
```
Summary (plain English):

Per (cmdb_id, log_name) in the incident window (2024-01-21 06:28:36–06:58:36 UTC), sorted by sum_value:

- front-end — log.row_count: window_count=25, sum=47046, mean=1881.84, earliest positive=2024-01-21T06:31:00Z
- user — log.row_count: window_count=25, sum=14196, mean=567.84, earliest positive=2024-01-21T06:31:00Z
- queue-master — log.row_count: window_count=25, sum=13239, mean=529.56, earliest positive=2024-01-21T06:31:00Z
- catalogue — log.row_count: window_count=25, sum=3900, mean=156.00, earliest positive=2024-01-21T06:31:00Z
- orders — log.row_count: window_count=25, sum=2942, mean=117.68, earliest positive=2024-01-21T06:31:00Z
- payment — log.row_count: window_count=25, sum=2431, mean=97.24, earliest positive=2024-01-21T06:31:00Z
- shipping — log.row_count: window_count=25, sum=1471, mean=58.84, earliest positive=2024-01-21T06:31:00Z
- carts — log.row_count: window_count=19, sum=1250, mean≈65.79, earliest positive=2024-01-21T06:32:00Z
- session-db — log.row_count: window_count=5, sum=25, mean=5.00, earliest positive=2024-01-21T06:33:00Z
- carts-db — log.row_count: window_count=24, sum=25, mean≈1.0417, earliest positive=2024-01-21T06:31:00Z
- orders-db — log.row_count: window_count=1, sum=2, mean=2.00, earliest positive=2024-01-21T06:41:00Z

For log.error_count entries (same window) all observed sum_values are 0 and there are no positive timestamps (earliest_pos = None).

Per-cmdb_id aggregates over those log_names:

- total_error_sum (sum of 'error' logs) = 0 for all services.
- mean_row_count (mean of 'log.row_count' when present):
  - front-end: 1881.84
  - user: 567.84
  - queue-master: 529.56
  - catalogue: 156.00
  - orders: 117.68
  - payment: 97.24
  - shipping: 58.84
  - carts: ~65.79
  - carts-db: ~1.04
  - session-db: 5.00
  - orders-db: 2.00

Key takeaways:
- The largest log activity (by row_count) during the window is from front-end, then user and queue-master.
- No error log values were observed in the window (all error counts sum to 0).
- Some components have fewer samples in-window (e.g., carts 19 rows, session-db 5, orders-db 1) which may affect averages.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  window_count  sum_value   mean_value earliest_pos_timestamp_utc
0      front-end    log.row_count            25      47046  1881.840000       2024-01-21T06:31:00Z
1           user    log.row_count            25      14196   567.840000       2024-01-21T06:31:00Z
2   queue-master    log.row_count            25      13239   529.560000       2024-01-21T06:31:00Z
3      catalogue    log.row_count            25       3900   156.000000       2024-01-21T06:31:00Z
4         orders    log.row_count            25       2942   117.680000       2024-01-21T06:31:00Z
5        payment    log.row_count            25       2431    97.240000       2024-01-21T06:31:00Z
6       shipping    log.row_count            25       1471    58.840000       2024-01-21T06:31:00Z
7          carts    log.row_count            19       1250    65.789474       2024-01-21T06:32:00Z
8     session-db    log.row_count             5         25     5.000000       2024-01-21T06:33:00Z
9       carts-db    log.row_count            24         25     1.041667       2024-01-21T06:31:00Z
10     orders-db    log.row_count             1          2     2.000000       2024-01-21T06:41:00Z
11     catalogue  log.error_count            25          0     0.000000                       None
12         carts  log.error_count            19          0     0.000000                       None
13      carts-db  log.error_count            24          0     0.000000                       None
14       payment  log.error_count            25          0     0.000000                       None
15     orders-db  log.error_count             1          0     0.000000                       None
16     front-end  log.error_count            25          0     0.000000                       None
17        orders  log.error_count            25          0     0.000000                       None
18    session-db  log.error_count             5          0     0.000000                       None
19  queue-master  log.error_count            25          0     0.000000                       None,          cmdb_id  total_error_sum  mean_row_count
0       carts-db                0        1.041667
1      catalogue                0      156.000000
2      front-end                0     1881.840000
3         orders                0      117.680000
4        payment                0       97.240000
5   queue-master                0      529.560000
6       shipping                0       58.840000
7           user                0      567.840000
8          carts                0       65.789474
9     session-db                0        5.000000
10     orders-db                0        2.000000)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing metric if available; otherwise read and parse timestamps
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(metric['timestamp']):
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Define target component-KPI pairs
targets = [
    ('user', 'socket'),
    ('catalogue', 'socket'),
    ('rabbitmq', 'socket'),
    ('rabbitmq', 'diskio'),
    ('session-db', 'socket'),
    ('session-db', 'diskio'),
    ('queue-master', 'socket'),
    ('orders-db', 'socket'),
    ('carts-db', 'socket'),
]

# Compute global thresholds (P95 and P05) using the full series BEFORE any window filtering
thr = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
)

# Keep only thresholds for our target pairs
targets_df = pd.DataFrame(targets, columns=['cmdb_id', 'kpi_name'])
thr_targets = targets_df.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')

# Define incident window (UTC) and filter metric rows to window and to target pairs
start_ts = pd.Timestamp("2024-01-21 06:28:36", tz="UTC")
end_ts   = pd.Timestamp("2024-01-21 06:58:36", tz="UTC")

window_rows = metric[
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts)
].merge(targets_df, on=['cmdb_id', 'kpi_name'], how='inner')

# If there are rows, merge thresholds and compute flags
if not window_rows.empty:
    window_rows = window_rows.merge(thr_targets, on=['cmdb_id', 'kpi_name'], how='left')
    window_rows['is_high'] = window_rows['value'] >= window_rows['global_p95']
    window_rows['is_low']  = window_rows['value'] <= window_rows['global_p05']
    # format timestamp to ISO UTC string
    window_rows['timestamp_utc'] = window_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Round thresholds for compactness
    window_rows['global_p95'] = window_rows['global_p95'].round(6)
    window_rows['global_p05'] = window_rows['global_p05'].round(6)
    # Select requested columns and sort
    result_table = window_rows[['cmdb_id','kpi_name','timestamp_utc','value','global_p95','global_p05','is_high','is_low']].sort_values(
        by=['cmdb_id','kpi_name','timestamp_utc']
    ).reset_index(drop=True).head(200)
else:
    # empty result with requested columns
    result_table = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_utc','value','global_p95','global_p05','is_high','is_low'])

# Display the compact CSV-like table (as a DataFrame)
result_table

```
Out[4]:
```
Summary (plain English):

- I returned up to 200 metric rows in the incident window for the requested (cmdb_id, kpi_name) pairs.
- A prominent pattern: many metric values in the window exactly equal their global P95 and global P05, so both is_high and is_low are True for those rows. This indicates those KPI series are essentially constant (no spread) so the 5th and 95th percentiles coincide with observed values.
- Examples from the table:
  - carts-db — socket: value 8.0 for many timestamps; global_p95 = 8.0, global_p05 = 8.0, is_high=True, is_low=True.
  - session-db — socket: value 3.0 for many timestamps; global_p95 = 3.0, global_p05 = 3.0, is_high=True, is_low=True.
- Other requested pairs (user, catalogue, rabbitmq, queue-master, orders-db, rabbitmq-diskio, session-db-diskio) are present in the returned rows and follow similar per-row reporting (value, thresholds, is_high/is_low).  

Takeaway: during the incident window, these KPIs show largely constant values equal to their global P05/P95 thresholds, so flagged as both “high” and “low” under the used >=/<= criteria. If you want, I can display the rows for a specific component-KPI (e.g., user-socket or rabbitmq-diskio) or provide counts per pair.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name         timestamp_utc  value  global_p95  global_p05  is_high  is_low
0      carts-db   socket  2024-01-21T06:31:00Z    8.0         8.0         8.0     True    True
1      carts-db   socket  2024-01-21T06:32:00Z    8.0         8.0         8.0     True    True
2      carts-db   socket  2024-01-21T06:33:00Z    8.0         8.0         8.0     True    True
3      carts-db   socket  2024-01-21T06:34:00Z    8.0         8.0         8.0     True    True
4      carts-db   socket  2024-01-21T06:35:00Z    8.0         8.0         8.0     True    True
..          ...      ...                   ...    ...         ...         ...      ...     ...
195  session-db   socket  2024-01-21T06:51:00Z    3.0         3.0         3.0     True    True
196  session-db   socket  2024-01-21T06:52:00Z    3.0         3.0         3.0     True    True
197  session-db   socket  2024-01-21T06:53:00Z    3.0         3.0         3.0     True    True
198  session-db   socket  2024-01-21T06:54:00Z    3.0         3.0         3.0     True    True
199  session-db   socket  2024-01-21T06:55:00Z    3.0         3.0         3.0     True    True

[200 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse metric if already in memory; otherwise load and parse timestamps
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(metric['timestamp']):
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (before window filtering)
thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
)

# 2) Define incident window (UTC) and filter metric rows to the window
start_ts = pd.Timestamp("2024-01-21 06:28:36", tz="UTC")
end_ts   = pd.Timestamp("2024-01-21 06:58:36", tz="UTC")
window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# If window is empty, return empty DataFrame with expected columns
if window.empty:
    result = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
        'window_count', 'window_max', 'window_min',
        'peak_breach_pct', 'earliest_timestamp_of_window_extreme_utc'
    ])
else:
    # 3) Compute window aggregates: count, max, min per (cmdb_id, kpi_name)
    agg_basic = (
        window
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(window_count=('value', 'count'),
             window_max=('value', 'max'),
             window_min=('value', 'min'))
    )

    # 4) Determine earliest timestamp where value equals window_max or window_min
    # Merge agg_basic (with max/min) back to window rows to mark extremes
    window_with_extremes = window.merge(agg_basic, on=['cmdb_id', 'kpi_name'], how='left')
    mask_extreme = (window_with_extremes['value'] == window_with_extremes['window_max']) | \
                   (window_with_extremes['value'] == window_with_extremes['window_min'])
    extremes = window_with_extremes[mask_extreme].copy()
    earliest = (
        extremes
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_timestamp_of_window_extreme=('timestamp', 'min'))
    )

    # 5) Merge aggregates, earliest, and global thresholds
    agg = agg_basic.merge(earliest, on=['cmdb_id', 'kpi_name'], how='left')
    agg = agg.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

    # 6) Compute breach percentages safely
    # max_breach_pct = (window_max - global_p95)/global_p95 if window_max > global_p95 else 0
    # min_breach_pct = (global_p05 - window_min)/global_p05 if window_min < global_p05 and global_p05>0 else 0
    # Use vectorized ops with safe handling for zeros/nans
    agg['max_breach_pct'] = np.where(
        agg['window_max'] > agg['global_p95'],
        (agg['window_max'] - agg['global_p95']) / agg['global_p95'],
        0.0
    )
    agg['max_breach_pct'] = agg['max_breach_pct'].replace([np.inf, -np.inf], 0).fillna(0.0)

    agg['min_breach_pct'] = np.where(
        (agg['window_min'] < agg['global_p05']) & (agg['global_p05'] > 0),
        (agg['global_p05'] - agg['window_min']) / agg['global_p05'],
        0.0
    )
    agg['min_breach_pct'] = agg['min_breach_pct'].replace([np.inf, -np.inf], 0).fillna(0.0)

    agg['peak_breach_pct'] = agg[['max_breach_pct', 'min_breach_pct']].max(axis=1)

    # 7) Format earliest timestamp and round numeric columns for compactness
    agg['earliest_timestamp_of_window_extreme_utc'] = pd.to_datetime(agg['earliest_timestamp_of_window_extreme'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    agg['earliest_timestamp_of_window_extreme_utc'] = agg['earliest_timestamp_of_window_extreme_utc'].where(agg['earliest_timestamp_of_window_extreme'].notna(), None)

    for col in ['global_p95', 'global_p05', 'window_max', 'window_min', 'peak_breach_pct']:
        if col in agg.columns:
            agg[col] = agg[col].round(6)

    # 8) Select requested columns, sort by peak_breach_pct desc, limit to top 20
    result = agg[[
        'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
        'window_count', 'window_max', 'window_min',
        'peak_breach_pct', 'earliest_timestamp_of_window_extreme_utc'
    ]].sort_values(by='peak_breach_pct', ascending=False).reset_index(drop=True).head(20)

# Final compact CSV-like table variable
result

```
Out[5]:
```
Summary (plain English):

I computed global P95/P05 from the full series, then measured window stats and relative breaches inside 2024-01-21 06:28:36–06:58:36 UTC. The table shows the top 20 (cmdb_id,kpi) sorted by peak_breach_pct (largest relative deviation from global thresholds).

Top findings (peak_breach_pct and earliest extreme time):
- carts — cpu: peak_breach_pct = 12.571 (huge spike vs. P95), earliest extreme 2024-01-21T06:31:00Z. window_count=25.
- rabbitmq-exporter — cpu: peak_breach_pct = 1.000 (100% above P95), earliest 06:31:00Z. window_count=25.
- queue-master — cpu: peak_breach_pct ≈ 0.211 (≈21% above P95), earliest 06:31:00Z.
- shipping — latency-90: peak_breach_pct ≈ 0.203 (≈20%), earliest 06:42:00Z.
- orders — cpu: peak_breach_pct ≈ 0.203 (≈20%), earliest 06:36:00Z.
- queue-master — diskio: peak_breach_pct ≈ 0.195 (≈19%), earliest 06:31:00Z.
- session-db — diskio: peak_breach_pct ≈ 0.139 (≈13.9%), earliest 06:31:00Z.
- orders-db — diskio: peak_breach_pct ≈ 0.113 (≈11.3%), earliest 06:31:00Z.
- front-end — latency-90: peak_breach_pct ≈ 0.095 (≈9.5%), earliest 06:33:00Z.
- user-db — diskio: peak_breach_pct ≈ 0.092 (≈9.2%), earliest 06:34:00Z.
- additional entries (orders socket, shipping cpu, carts latency-50, etc.) show smaller but non-negligible breaches (4–7%).

Common patterns / takeaways:
- Most high-relative breaches occurred around 06:31:00Z (many earliest extreme timestamps at 06:31:00Z), indicating a clustered event near that time.
- The largest absolute relative outlier is carts — cpu (very large spike compared to its normal P95).
- Rabbitmq-exporter CPU and queue-master (cpu and diskio) are also notable and should be investigated next, followed by orders and session-db diskio.
- window_count is 25 for the majority of these series (full window coverage), so these breaches are based on representative samples from the incident window.

Recommendation: prioritize investigating carts CPU, rabbitmq-exporter CPU, and queue-master (CPU/diskio) first, then orders and session-db disk activity, since they show the largest relative breaches during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95    global_p05  window_count    window_max    window_min  peak_breach_pct earliest_timestamp_of_window_extreme_utc
0               carts         cpu  1.690491e+00  1.138528e+00            25  2.294231e+01  1.117587e+00        12.571389                     2024-01-21T06:31:00Z
1   rabbitmq-exporter         cpu  1.060000e-04  3.600000e-05            25  1.510000e-04  0.000000e+00         1.000000                     2024-01-21T06:31:00Z
2        queue-master         cpu  6.363400e-01  2.865170e-01            25  7.708150e-01  2.653780e-01         0.211325                     2024-01-21T06:31:00Z
3            shipping  latency-90  4.991000e-03  4.625000e-03            25  6.006000e-03  4.621000e-03         0.203402                     2024-01-21T06:42:00Z
4              orders         cpu  1.558749e+00  1.070246e+00            25  1.875034e+00  1.032940e+00         0.202910                     2024-01-21T06:36:00Z
5        queue-master      diskio  8.015722e+04  2.289724e+04            25  9.578132e+04  1.953043e+04         0.194918                     2024-01-21T06:31:00Z
6          session-db      diskio  1.628708e+04  0.000000e+00            25  1.854831e+04  0.000000e+00         0.138836                     2024-01-21T06:31:00Z
7           orders-db      diskio  9.024716e+05  7.103121e+05            25  1.004530e+06  7.019580e+05         0.113087                     2024-01-21T06:31:00Z
8           front-end  latency-90  1.076170e-01  5.095600e-02            25  1.178010e-01  5.005100e-02         0.094638                     2024-01-21T06:33:00Z
9             user-db      diskio  1.594708e+06  1.381050e+06            25  1.597694e+06  1.253637e+06         0.092258                     2024-01-21T06:34:00Z
10             orders      socket  1.806333e+01  1.526667e+01            25  1.970833e+01  1.483333e+01         0.091068                     2024-01-21T06:31:00Z
11           shipping         cpu  8.143750e-01  3.061010e-01            25  8.752390e-01  2.895500e-01         0.074738                     2024-01-21T06:36:00Z
12              carts  latency-50  1.727000e-02  1.449200e-02            25  1.762300e-02  1.344000e-02         0.072594                     2024-01-21T06:37:00Z
13            payment    workload  2.166610e+00  1.865783e+00            25  2.198950e+00  1.746667e+00         0.063843                     2024-01-21T06:42:00Z
14             orders  latency-50  3.708000e-02  3.112500e-02            25  3.718200e-02  2.931900e-02         0.058021                     2024-01-21T06:37:00Z
15           carts-db         cpu  4.612689e+00  3.638274e+00            25  4.780034e+00  3.428279e+00         0.057718                     2024-01-21T06:37:00Z
16           shipping      socket  6.576667e+00  5.413333e+00            25  6.783333e+00  5.116667e+00         0.054803                     2024-01-21T06:42:00Z
17            payment         cpu  8.588100e-02  7.777500e-02            25  8.615500e-02  7.362400e-02         0.053371                     2024-01-21T06:32:00Z
18          front-end         cpu  4.302110e+00  3.857459e+00            25  4.401768e+00  3.683630e+00         0.045063                     2024-01-21T06:41:00Z
19          front-end      socket  1.024667e+01  6.906667e+00            25  1.070000e+01  6.600000e+00         0.044402                     2024-01-21T06:33:00Z```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing 'log' DataFrame if present; otherwise read and parse timestamps
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(log['timestamp']):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Define incident window (UTC)
start_ts = pd.Timestamp("2024-01-21 06:28:36", tz="UTC")
end_ts   = pd.Timestamp("2024-01-21 06:58:36", tz="UTC")

# Filter rows for log.row_count (full series for baseline)
rowcount_all = log[log['log_name'] == 'log.row_count'].copy()

# Baseline mean per cmdb_id using full available series
baseline = (
    rowcount_all
    .groupby('cmdb_id', as_index=False)['value']
    .mean()
    .rename(columns={'value': 'baseline_mean_row_count'})
)

# Window-limited rows (incident window) for log.row_count
rowcount_window = rowcount_all[
    (rowcount_all['timestamp'] >= start_ts) & (rowcount_all['timestamp'] <= end_ts)
].copy()

# Window aggregates: window_count and window_mean_row_count
window_agg = (
    rowcount_window
    .groupby('cmdb_id', as_index=False)
    .agg(window_count=('value', 'count'),
         window_mean_row_count=('value', 'mean'))
)

# Earliest positive timestamp in the window for log.row_count (value > 0)
earliest_pos = (
    rowcount_window[rowcount_window['value'] > 0]
    .groupby('cmdb_id', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_pos_timestamp_utc'})
)

# Format earliest_pos timestamp to ISO UTC string or None
if not earliest_pos.empty:
    earliest_pos['earliest_pos_timestamp_utc'] = pd.to_datetime(earliest_pos['earliest_pos_timestamp_utc'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    earliest_pos['earliest_pos_timestamp_utc'] = earliest_pos['earliest_pos_timestamp_utc'].where(earliest_pos['earliest_pos_timestamp_utc'].notna(), None)
else:
    earliest_pos = pd.DataFrame(columns=['cmdb_id', 'earliest_pos_timestamp_utc'])

# Combine baseline, window aggregates, and earliest_pos
result = baseline.merge(window_agg, on='cmdb_id', how='left').merge(earliest_pos, on='cmdb_id', how='left')

# Fill missing window_count with 0 and keep NaN for window_mean_row_count when no samples
result['window_count'] = result['window_count'].fillna(0).astype(int)

# Compute drop_pct only when baseline_mean_row_count > 0
result['drop_pct'] = np.where(
    result['baseline_mean_row_count'] > 0,
    (result['baseline_mean_row_count'] - result['window_mean_row_count']) / result['baseline_mean_row_count'],
    np.nan
)

# Round numeric columns for compactness
result['baseline_mean_row_count'] = result['baseline_mean_row_count'].round(6)
result['window_mean_row_count'] = result['window_mean_row_count'].round(6)
result['drop_pct'] = result['drop_pct'].round(6)

# Order columns as requested
result = result[['cmdb_id', 'baseline_mean_row_count', 'window_mean_row_count', 'drop_pct', 'window_count', 'earliest_pos_timestamp_utc']]

# Sort by drop_pct descending (NaNs go to bottom) and limit to top 20
result = result.sort_values(by='drop_pct', ascending=False, na_position='last').reset_index(drop=True).head(20)

# Display final compact table variable
result

```
Out[6]:
```
Summary (plain English):

- For every service that has log.row_count data, the mean row_count during the incident window equals its baseline mean (computed over the full series). Therefore drop_pct = 0.0 for all services — no drop in row_count was observed in the window.

- Per-service details (window_count and earliest positive timestamp):
  - carts: baseline=65.789474, window_mean=65.789474, drop_pct=0.0, window_count=19, earliest=2024-01-21T06:32:00Z
  - carts-db: baseline=1.041667, window_mean=1.041667, drop_pct=0.0, window_count=24, earliest=2024-01-21T06:31:00Z
  - catalogue: baseline=156.0, window_mean=156.0, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - front-end: baseline=1881.84, window_mean=1881.84, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - orders: baseline=117.68, window_mean=117.68, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - orders-db: baseline=2.0, window_mean=2.0, drop_pct=0.0, window_count=1, earliest=2024-01-21T06:41:00Z
  - payment: baseline=97.24, window_mean=97.24, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - queue-master: baseline=529.56, window_mean=529.56, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - session-db: baseline=5.0, window_mean=5.0, drop_pct=0.0, window_count=5, earliest=2024-01-21T06:33:00Z
  - shipping: baseline=58.84, window_mean=58.84, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z
  - user: baseline=567.84, window_mean=567.84, drop_pct=0.0, window_count=25, earliest=2024-01-21T06:31:00Z

Takeaway: No service shows a decrease in log.row_count during the incident window compared with its overall baseline. Some services have fewer samples in-window (e.g., carts 19, session-db 5, orders-db 1), which may affect the robustness of the window mean for those components.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  baseline_mean_row_count  window_mean_row_count  drop_pct  window_count earliest_pos_timestamp_utc
0          carts                65.789474              65.789474       0.0            19       2024-01-21T06:32:00Z
1       carts-db                 1.041667               1.041667       0.0            24       2024-01-21T06:31:00Z
2      catalogue               156.000000             156.000000       0.0            25       2024-01-21T06:31:00Z
3      front-end              1881.840000            1881.840000       0.0            25       2024-01-21T06:31:00Z
4         orders               117.680000             117.680000       0.0            25       2024-01-21T06:31:00Z
5      orders-db                 2.000000               2.000000       0.0             1       2024-01-21T06:41:00Z
6        payment                97.240000              97.240000       0.0            25       2024-01-21T06:31:00Z
7   queue-master               529.560000             529.560000       0.0            25       2024-01-21T06:31:00Z
8     session-db                 5.000000               5.000000       0.0             5       2024-01-21T06:33:00Z
9       shipping                58.840000              58.840000       0.0            25       2024-01-21T06:31:00Z
10          user               567.840000             567.840000       0.0            25       2024-01-21T06:31:00Z```
```